**Classification of Gout from Chief Complaints in the Emergency Department:**

**Comparison of Feature-based and Representation Learning Approaches**

This notebook contains the code for the LHS 712 Final Project completed on April 27th, 2026.





In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.listdir("/content/drive/MyDrive/")

!pip install imbalanced-learn



In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from collections import Counter
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, StratifiedKFold
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, f1_score, precision_score,
    recall_score, accuracy_score, confusion_matrix, roc_auc_score,
    roc_curve
)
from sklearn.calibration import CalibratedClassifierCV

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

warnings.filterwarnings("ignore")

In [ ]:

################################################################################
# 1) CONFIGURATION #############################################################
################################################################################

CONFIG = {

    "DATA_PATH_2019": "/content/drive/MyDrive/Colab Notebooks/GOUT-CC-2019-CORPUS-REDACTED.csv",
    "DATA_PATH_2020": "/content/drive/MyDrive/Colab Notebooks/GOUT-CC-2020-CORPUS-REDACTED.csv",

    # set column names to what we want them to point to
    "TEXT_COL":  "Chief Complaint",
    "LABEL_COL": "Consensus",

    # specify the split ratios (70/15/15) and random seed
    "TRAIN_SIZE":  0.70,
    "VAL_SIZE":    0.15,
    "TEST_SIZE":   0.15,
    "RANDOM_SEED": 107,

    # BERT/RoBERTa settings
    "BERT_MODEL":     "bert-base-uncased",
    "CLINICAL_MODEL": "emilyalsentzer/Bio_ClinicalBERT",
    "ROBERTA_MODEL":  "allenai/biomed_roberta_base",
    "MAX_LEN":     128,
    "BATCH_SIZE":   32,
    "BERT_EPOCHS":   5,
    "BERT_LR":     1e-5,

    # set the output directory
    "OUTPUT_DIR": "/content/drive/MyDrive/gout_outputs",
}

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")




In [ ]:
################################################################################
# 2) DATA LOADING- COMBINED 2019 + 2020 #######################################
################################################################################

def load_data() -> pd.DataFrame:
    # read the data files
    df_2019 = pd.read_csv(CONFIG["DATA_PATH_2019"])
    df_2020 = pd.read_csv(CONFIG["DATA_PATH_2020"])
    # combine the data files
    df = pd.concat([df_2019, df_2020], ignore_index=True)

    print(f"Combined dataset: {len(df):,} rows")
    print(f"Raw Consensus counts:\n{df['Consensus'].value_counts()}")
    print(f"Raw Predict counts:\n{df['Predict'].value_counts()}")

    # specify the label column *consensus) and also the predict column for resolving label discrepancies
    label_col   = CONFIG["LABEL_COL"]
    predict_col = "Predict"

    df[label_col]   = df[label_col].astype(str).str.strip().str.upper()
    df[predict_col] = df[predict_col].astype(str).str.strip().str.upper()

    # function to
    def resolve_label(row):
        if row[label_col] == "Y":
            return 1
        elif row[label_col] == "N":
            return 0
        else:
            # when consensus is "U" or "-", use the predict column
            if row[predict_col] == "Y":
                return 1
            else:
                return 0  # "N", "U", and "-" all become NON cases

    df["label"] = df.apply(resolve_label, axis=1)
    CONFIG["LABEL_COL"] = "label"

    print(f"\nAfter fallback labeling: {len(df):,} rows")
    print(f"Label distribution:\n{df['label'].value_counts()}")
    ratio = df['label'].value_counts()[0] / df['label'].value_counts()[1]
    print(f"Imbalance ratio: {ratio:.1f}:1")
    return df

In [ ]:
################################################################################
# 3. EDA #######################################################################
################################################################################

def run_eda(df: pd.DataFrame, text_col: str, label_col: str) -> None:
    print(f"\n{'='*60}")
    print("EXPLORATORY DATA ANALYSIS")
    print(f"{'='*60}")

    # get the class distribution
    vc = df[label_col].value_counts()
    print("\nClass distribution:")
    for lbl, cnt in vc.items():
        name = "Gout" if lbl == 1 else "Non-Gout"
        print(f"  {name} ({lbl}): {cnt:,}  ({cnt/len(df)*100:.1f}%)")

    # Get the summary stats for the Chief Complaints column
    print("\nText length statistics (characters):")
    df = df.copy()
    df["text_len"] = df[text_col].astype(str).str.len()
    print(df.groupby(label_col)["text_len"].describe().round(1))

    print("\nTop 20 words (gout class):")
    gout_words = " ".join(df[df[label_col] == 1][text_col].astype(str)).lower()
    gout_words = re.findall(r'\b[a-z]{3,}\b', gout_words)
    stopwords  = {"the", "and", "for", "with", "this", "that", "from",
                  "have", "has", "was", "are", "were", "not", "but"}
    top_gout = Counter(w for w in gout_words if w not in stopwords).most_common(20)
    for word, count in top_gout:
        print(f"  {word}: {count}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("EDA: Gout ED Chief Complaint Corpus (2019+2020)",
                 fontsize=14, fontweight="bold")

    labels_text = ["Non-Gout", "Gout"]
    counts = [vc.get(0, 0), vc.get(1, 0)]
    colors = ["#4C72B0", "#DD8452"]
    axes[0].bar(labels_text, counts, color=colors, edgecolor="white", width=0.5)
    axes[0].set_title("Class Distribution")
    axes[0].set_ylabel("Count")
    for i, c in enumerate(counts):
        axes[0].text(i, c + max(counts) * 0.01, str(c), ha="center", fontsize=11)

    for lbl, color, name in zip([0, 1], colors, labels_text):
        subset = df[df[label_col] == lbl]["text_len"]
        axes[1].hist(subset, bins=40, alpha=0.7, color=color, label=name)
    axes[1].set_title("Text Length Distribution")
    axes[1].set_xlabel("Characters")
    axes[1].set_ylabel("Frequency")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"], "eda_plots.png"), dpi=150)
    plt.close()
    print("\nEDA plot saved.")

In [ ]:
################################################################################
# 4. PREPROCESSING #############################################################
################################################################################

# reg ex med abbreviation expansion for feature-based models
MEDICAL_ABBREVIATIONS = {
    r'\bjt\b':   'joint',
    r'\bpts?\b': 'patient',
    r'\bsob\b':  'shortness of breath',
    r'\bc/o\b':  'complains of',
    r'\bh/o\b':  'history of',
    r'(?<![a-z])knee(?![a-z])':    'knee',
    r'(?<![a-z])ankle(?![a-z])':   'ankle',
    r'(?<![a-z])wrist(?![a-z])':   'wrist',
    r'(?<![a-z])toe(?![a-z])':     'toe',
    r'(?<![a-z])elbow(?![a-z])':   'elbow',
    r'(?<![a-z])shoulder(?![a-z])':'shoulder',
    r'(?<![a-z])foot(?![a-z])':    'foot',
    r'(?<![a-z])hip(?![a-z])':     'hip',

    r'\bhtn\b':  'hypertension',
    r'\bdm\b':   'diabetes mellitus',
    r'\bdm2\b':  'type 2 diabetes',
    r'\bchf\b':  'congestive heart failure',
    r'\bckd\b':  'chronic kidney disease',
    r'\bcad\b':  'coronary artery disease',
    r'\bafib\b': 'atrial fibrillation',
    r'\bgerd\b': 'gastroesophageal reflux disease',
    r'\bcopd\b': 'chronic obstructive pulmonary disease',
    r'\bobe\b':  'obesity',
    r'\bra\b':   'rheumatoid arthritis',
    r'\boa\b':   'osteoarthritis'
}

def preprocess_text(text: str) -> str:
    text = str(text).lower().strip()
    for abbrev, expansion in MEDICAL_ABBREVIATIONS.items():
        text = re.sub(abbrev, expansion, text)
    text = re.sub(r'[^\w\s\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_text_no_expansion(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r'[^\w\s\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_dataframe(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    df = df.copy()
    df["clean_text"]          = df[text_col].apply(preprocess_text)
    df["clean_text_no_expand"] = df[text_col].apply(preprocess_text_no_expansion)
    return df

def split_data(df: pd.DataFrame, text_col: str, label_col: str):
    X = df[text_col].values
    y = df[label_col].values
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=(CONFIG["VAL_SIZE"] + CONFIG["TEST_SIZE"]),
        stratify=y,
        random_state=CONFIG["RANDOM_SEED"]
    )
    val_ratio = CONFIG["VAL_SIZE"] / (CONFIG["VAL_SIZE"] + CONFIG["TEST_SIZE"])
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=(1 - val_ratio),
        stratify=y_temp,
        random_state=CONFIG["RANDOM_SEED"]
    )
    print(f"\nSplit- Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    print(f"Gout in train: {y_train.sum()}, val: {y_val.sum()}, test: {y_test.sum()}")
    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
# ==============================================================================
# 5. EVALUATION UTILITIES
# ==============================================================================

def evaluate_model(model_name: str, y_true, y_pred, y_proba=None) -> dict:
    print(f"\n{'='*60}")
    print(f"RESULTS: {model_name}")
    print(f"{'='*60}")
    print(classification_report(y_true, y_pred, target_names=["Non-Gout", "Gout"]))

    results = {
        "model":          model_name,
        "accuracy":       round(accuracy_score(y_true, y_pred), 4),
        "f1_macro":       round(f1_score(y_true, y_pred, average="macro"), 4),
        "f1_weighted":    round(f1_score(y_true, y_pred, average="weighted"), 4),
        "f1_gout":        round(f1_score(y_true, y_pred, average="binary", pos_label=1), 4),
        "precision_gout": round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        "recall_gout":    round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
    }
    if y_proba is not None:
        results["roc_auc"] = round(roc_auc_score(y_true, y_proba), 4)

    print(f"  F1 (Gout):        {results['f1_gout']:.4f}")
    print(f"  Precision (Gout): {results['precision_gout']:.4f}")
    print(f"  Recall (Gout):    {results['recall_gout']:.4f}")
    if y_proba is not None:
        print(f"  ROC-AUC:          {results['roc_auc']:.4f}")
    return results

def plot_confusion_matrix(model_name: str, y_true, y_pred, save_prefix: str) -> None:
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Non-Gout", "Gout"],
                yticklabels=["Non-Gout", "Gout"], ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion Matrix- {model_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"], f"cm_{save_prefix}.png"), dpi=150)
    plt.close()

In [ ]:
# ==============================================================================
# 6. MODEL 1- LINEAR SVM + TF-IDF + SMOTE
# ==============================================================================

def train_svm(X_train, y_train, X_val, y_val) -> tuple:
    print(f"\n{'='*60}")
    print("MODEL 1: Linear SVM + TF-IDF + SMOTE")
    print(f"{'='*60}")

    tfidf = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2),
                            max_features=15000)
    X_train_vec = tfidf.fit_transform(X_train)
    X_val_vec   = tfidf.transform(X_val)


    smote = SMOTE(random_state=CONFIG["RANDOM_SEED"])
    X_train_res, y_train_res = smote.fit_resample(X_train_vec, y_train)
    print(f"After SMOTE: {pd.Series(y_train_res).value_counts().to_dict()}")


    clf = CalibratedClassifierCV(
        LinearSVC(max_iter=2000, class_weight="balanced"), cv=3
    )
    param_grid = {"estimator__C": [0.01, 0.1, 1.0, 10.0]}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=CONFIG["RANDOM_SEED"])
    search = RandomizedSearchCV(clf, param_grid, n_iter=4, scoring="f1",
                                cv=cv, n_jobs=-1, verbose=1,
                                random_state=CONFIG["RANDOM_SEED"])
    search.fit(X_train_res, y_train_res)

    best_clf = search.best_estimator_
    print(f"Best C: {search.best_params_} | CV F1: {search.best_score_:.4f}")

    y_val_pred  = best_clf.predict(X_val_vec)
    y_val_proba = best_clf.predict_proba(X_val_vec)[:, 1]

    results = evaluate_model("Linear SVM + TF-IDF", y_val, y_val_pred, y_val_proba)
    plot_confusion_matrix("Linear SVM + TF-IDF", y_val, y_val_pred, "svm")

    return (tfidf, best_clf), results


In [ ]:
# ==============================================================================
# 7. MODEL 2- RANDOM FOREST + TF-IDF + SMOTE
# ==============================================================================

def train_random_forest(X_train, y_train, X_val, y_val) -> tuple:
    print(f"\n{'='*60}")
    print("MODEL 2: Random Forest + TF-IDF + SMOTE")
    print(f"{'='*60}")

    tfidf = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2),
                            max_features=15000)
    X_train_vec = tfidf.fit_transform(X_train)
    X_val_vec   = tfidf.transform(X_val)

    smote = SMOTE(random_state=CONFIG["RANDOM_SEED"])
    X_train_res, y_train_res = smote.fit_resample(X_train_vec, y_train)
    print(f"After SMOTE: {pd.Series(y_train_res).value_counts().to_dict()}")

    param_grid = {
        "n_estimators":    [100, 300, 500],
        "max_depth":       [None, 20, 50],
        "min_samples_split": [2, 5],
        "class_weight":    ["balanced"],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=CONFIG["RANDOM_SEED"])
    clf = RandomForestClassifier(n_jobs=-1, random_state=CONFIG["RANDOM_SEED"])
    search = RandomizedSearchCV(clf, param_grid, n_iter=20, scoring="f1",
                                cv=cv, n_jobs=-1, verbose=1,
                                random_state=CONFIG["RANDOM_SEED"])
    search.fit(X_train_res, y_train_res)

    best_clf = search.best_estimator_
    print(f"Best params: {search.best_params_} | CV F1: {search.best_score_:.4f}")

    y_val_pred  = best_clf.predict(X_val_vec)
    y_val_proba = best_clf.predict_proba(X_val_vec)[:, 1]

    results = evaluate_model("Random Forest + TF-IDF", y_val, y_val_pred, y_val_proba)
    plot_confusion_matrix("Random Forest + TF-IDF", y_val, y_val_pred, "rf")

    # rf feature importnance
    feature_names = tfidf.get_feature_names_out()
    importances   = best_clf.feature_importances_
    top_idx = np.argsort(importances)[-20:]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(feature_names[top_idx], importances[top_idx], color="#4C72B0")
    ax.set_title("Random Forest- Top 20 Feature Importances")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"], "rf_feature_importance.png"), dpi=150)
    plt.close()

    return (tfidf, best_clf), results


# ==============================================================================
# HELPER: Evaluate sklearn (tfidf, clf) tuple on test set
# ==============================================================================

def evaluate_sklearn_on_test(model_name, tfidf_clf_tuple, X_test, y_test) -> dict:
    tfidf, clf = tfidf_clf_tuple
    X_test_vec  = tfidf.transform(X_test)
    y_pred      = clf.predict(X_test_vec)
    y_proba     = clf.predict_proba(X_test_vec)[:, 1]
    results = evaluate_model(f"{model_name} [TEST]", y_test, y_pred, y_proba)
    plot_confusion_matrix(f"{model_name} (Test)", y_test, y_pred,
                          f"{model_name.replace(' ', '_').lower()}_test")
    return results

def get_sklearn_proba_on_test(tfidf_clf_tuple, X_test):
    tfidf, clf = tfidf_clf_tuple
    return clf.predict_proba(tfidf.transform(X_test))[:, 1]

def get_sklearn_proba_on_val(tfidf_clf_tuple, X_val):
    tfidf, clf = tfidf_clf_tuple
    return clf.predict_proba(tfidf.transform(X_val))[:, 1]


In [ ]:
# ==============================================================================
# 8. BERT DATASET
# ==============================================================================

class ChiefComplaintDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len: int):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(int(self.labels[idx]), dtype=torch.long),
        }

In [ ]:
# ==============================================================================
# 9. BERT/ROBERTA TRAINING
# ==============================================================================

def train_bert_model(model_name, display_name, save_prefix,
                     X_train, y_train, X_val, y_val) -> tuple:
    print(f"\n{'='*60}")
    print(f"MODEL: {display_name}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2
    )
    model.to(DEVICE)

    train_dataset = ChiefComplaintDataset(X_train, y_train, tokenizer, CONFIG["MAX_LEN"])
    val_dataset   = ChiefComplaintDataset(X_val,   y_val,   tokenizer, CONFIG["MAX_LEN"])
    train_loader  = DataLoader(train_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True)
    val_loader    = DataLoader(val_dataset,   batch_size=CONFIG["BATCH_SIZE"], shuffle=False)

    optimizer = AdamW(model.parameters(), lr=CONFIG["BERT_LR"], weight_decay=0.01)


    n_neg = int((y_train == 0).sum())
    n_pos = int((y_train == 1).sum())
    pos_weight = n_neg / n_pos
    # max weight at 10
    pos_weight = min(pos_weight, 10.0)
    class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float).to(DEVICE)
    print(f"Class weights- Non-Gout: 1.0, Gout: {pos_weight:.1f}")
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

    total_steps = len(train_loader) * CONFIG["BERT_EPOCHS"]
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )

    history    = {"train_loss": [], "val_f1": []}
    best_val_f1 = -1.0
    best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    best_preds, best_proba, best_labels = [], [], []

    for epoch in range(1, CONFIG["BERT_EPOCHS"] + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                            labels=labels)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        model.eval()
        all_preds, all_labels, all_proba = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels         = batch["labels"].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                probs   = torch.softmax(outputs.logits, dim=1)[:, 1]
                preds   = torch.argmax(outputs.logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_proba.extend(probs.cpu().numpy())

        val_f1 = f1_score(all_labels, all_preds, average="binary", pos_label=1)
        history["train_loss"].append(avg_loss)
        history["val_f1"].append(val_f1)
        print(f"  Epoch {epoch}/{CONFIG['BERT_EPOCHS']} | "
              f"Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_preds  = all_preds[:]
            best_proba  = all_proba[:]
            best_labels = all_labels[:]

    model.load_state_dict(best_state)
    model_path = os.path.join(CONFIG["OUTPUT_DIR"], f"{save_prefix}_best.pt")
    torch.save(model.state_dict(), model_path)
    print(f"Best model saved → {model_path}")

    # Training curve
    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax2 = ax1.twinx()
    ax1.plot(range(1, CONFIG["BERT_EPOCHS"] + 1), history["train_loss"],
             "b-o", label="Train Loss")
    ax2.plot(range(1, CONFIG["BERT_EPOCHS"] + 1), history["val_f1"],
             "r-s", label="Val F1")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss", color="b")
    ax2.set_ylabel("Val F1", color="r")
    ax1.set_title(f"Training Curve- {display_name}")
    lines1, l1 = ax1.get_legend_handles_labels()
    lines2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, l1 + l2, loc="upper right")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"], f"curve_{save_prefix}.png"), dpi=150)
    plt.close()

    results = evaluate_model(display_name, best_labels, best_preds, best_proba)
    plot_confusion_matrix(display_name, best_labels, best_preds, save_prefix)
    return model, tokenizer, results


def evaluate_bert_on_test(display_name, save_prefix, model, tokenizer,
                           X_test, y_test) -> tuple:
    model.eval()
    dataset = ChiefComplaintDataset(X_test, y_test, tokenizer, CONFIG["MAX_LEN"])
    loader  = DataLoader(dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False)
    all_preds, all_labels, all_proba = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"]
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs   = torch.softmax(outputs.logits, dim=1)[:, 1]
            preds   = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_proba.extend(probs.cpu().numpy())
    results = evaluate_model(f"{display_name} [TEST]", all_labels, all_preds, all_proba)
    plot_confusion_matrix(f"{display_name} (Test)", all_labels, all_preds,
                          f"{save_prefix}_test")
    return results, all_proba, all_labels

In [ ]:
# ==============================================================================
# 10. COMPARISON PLOTS
# ==============================================================================

def plot_model_comparison(all_results: list) -> None:
    metrics = ["f1_gout", "precision_gout", "recall_gout", "roc_auc"]
    labels  = [r["model"] for r in all_results]
    x       = np.arange(len(labels))
    width   = 0.18
    colors  = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

    fig, ax = plt.subplots(figsize=(16, 6))
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        vals = [r.get(metric, 0) for r in all_results]
        bars = ax.bar(x + i * width, vals, width,
                      label=metric.replace("_", " ").title(),
                      color=color, alpha=0.85)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005,
                    f"{bar.get_height():.3f}",
                    ha="center", va="bottom", fontsize=6.5, rotation=45)

    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.set_title("Model Comparison- Validation Set (Gout Class)", fontsize=13)
    ax.legend(loc="upper right")
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.2f"))
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"], "model_comparison.png"), dpi=150)
    plt.close()
    print("Comparison plot saved.")


def plot_roc_curves(roc_data: list) -> None:
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]
    for (name, fpr, tpr, auc_val), color in zip(roc_data, colors):
        ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc_val:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curves- All Models")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"], "roc_curves.png"), dpi=150)
    plt.close()
    print("ROC curve plot saved.")


In [ ]:
# ==============================================================================
# 11. THRESHOLD ANALYSIS
# ==============================================================================

def threshold_analysis(model_name: str, y_true, y_proba, save_prefix: str) -> None:
    thresholds = np.linspace(0.05, 0.95, 100)
    precisions, recalls, f1s = [], [], []
    for t in thresholds:
        preds = (np.array(y_proba) >= t).astype(int)
        if preds.sum() == 0:
            precisions.append(0); recalls.append(0); f1s.append(0)
            continue
        precisions.append(precision_score(y_true, preds, zero_division=0))
        recalls.append(recall_score(y_true, preds, zero_division=0))
        f1s.append(f1_score(y_true, preds, zero_division=0))

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(thresholds, precisions, label="Precision", color="#4C72B0")
    ax.plot(thresholds, recalls,    label="Recall",    color="#DD8452")
    ax.plot(thresholds, f1s,        label="F1",        color="#55A868")
    ax.axvline(0.5, color="gray", linestyle="--", linewidth=1, label="Default (0.5)")
    best_t = thresholds[np.argmax(f1s)]
    ax.axvline(best_t, color="red", linestyle=":", linewidth=1.5,
               label=f"Best F1 threshold ({best_t:.2f})")
    ax.set_xlabel("Decision Threshold")
    ax.set_ylabel("Score")
    ax.set_title(f"Threshold Analysis- {model_name}")
    ax.legend(loc="center right")
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"], f"threshold_{save_prefix}.png"), dpi=150)
    plt.close()
    print(f"Threshold plot saved. Optimal threshold for F1: {best_t:.2f}")


In [ ]:
# ==============================================================================
# 12. RESULTS EXPORT
# ==============================================================================

def save_results(val_results: list, test_results: list) -> None:
    val_df  = pd.DataFrame(val_results)
    test_df = pd.DataFrame(test_results)
    test_df.columns = [c if c == "model" else f"test_{c}" for c in test_df.columns]

    combined = val_df.merge(test_df, on="model")
    combined.to_csv(os.path.join(CONFIG["OUTPUT_DIR"], "results_summary.csv"), index=False)

    md_lines = ["# Model Results Summary\n",
                "## Validation Set\n",
                val_df.to_markdown(index=False),
                "\n\n## Test Set\n",
                test_df.to_markdown(index=False)]

    best_row = val_df.loc[val_df["f1_gout"].idxmax()]
    md_lines += [
        f"\n\n## Best Model\n",
        f"**{best_row['model']}** achieved the highest gout-class F1 on the "
        f"validation set: `{best_row['f1_gout']:.4f}`\n",
        "For a clinical alert system, prioritize **recall** to minimise missed gout cases. "
        "Consider deploying the best model with a lowered decision threshold "
        "(see threshold analysis plots) to trade some precision for higher recall.\n"
    ]

    with open(os.path.join(CONFIG["OUTPUT_DIR"], "results_summary.md"), "w") as f:
        f.write("\n".join(md_lines))

    print(f"\nResults saved to {CONFIG['OUTPUT_DIR']}")
    print("\nFull results table:")
    print(val_df.to_string(index=False))


In [ ]:
# ==============================================================================
# 13. MAIN PIPELINE
# ==============================================================================

def main():
    print("\n" + "#" * 60)
    print("GOUT ED CHIEF COMPLAINT- NLP CLASSIFICATION PIPELINE")
    print("#" * 60)

    # Load combined 2019 + 2020
    df = load_data()

    assert CONFIG["TEXT_COL"]  in df.columns, f"Column '{CONFIG['TEXT_COL']}' not found."
    assert CONFIG["LABEL_COL"] in df.columns, f"Column '{CONFIG['LABEL_COL']}' not found."

    run_eda(df, CONFIG["TEXT_COL"], CONFIG["LABEL_COL"])

    df = preprocess_dataframe(df, CONFIG["TEXT_COL"])

    # For SVM, Random Forest, and general BERT- with abbreviation expansion
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(
        df, "clean_text", CONFIG["LABEL_COL"]
    )

    # For ClinicalBERT and BioMed-RoBERTa- without abbreviation expansion
    X_train_ne, X_val_ne, X_test_ne, _, _, _ = split_data(
        df, "clean_text_no_expand", CONFIG["LABEL_COL"]
    )

    val_results, test_results, roc_data = [], [], []


    # ------------------------------------------------------------------
    # Model 1: SVM
    # ------------------------------------------------------------------
    svm_tuple, svm_val = train_svm(X_train, y_train, X_val, y_val)
    val_results.append(svm_val)

    svm_test = evaluate_sklearn_on_test("SVM", svm_tuple, X_test, y_test)
    test_results.append(svm_test)

    svm_proba_test = get_sklearn_proba_on_test(svm_tuple, X_test)
    fpr, tpr, _ = roc_curve(y_test, svm_proba_test)
    roc_data.append(("SVM", fpr, tpr, roc_auc_score(y_test, svm_proba_test)))

    threshold_analysis("Linear SVM", y_val,
                       get_sklearn_proba_on_val(svm_tuple, X_val), "svm")

    # ------------------------------------------------------------------
    # Model 2: Random Forest
    # ------------------------------------------------------------------
    rf_tuple, rf_val = train_random_forest(X_train, y_train, X_val, y_val)
    val_results.append(rf_val)

    rf_test = evaluate_sklearn_on_test("Random Forest", rf_tuple, X_test, y_test)
    test_results.append(rf_test)

    rf_proba_test = get_sklearn_proba_on_test(rf_tuple, X_test)
    fpr, tpr, _ = roc_curve(y_test, rf_proba_test)
    roc_data.append(("Random Forest", fpr, tpr, roc_auc_score(y_test, rf_proba_test)))

    threshold_analysis("Random Forest", y_val,
                       get_sklearn_proba_on_val(rf_tuple, X_val), "rf")

    # ------------------------------------------------------------------
    # Model 3: BERT
    # ------------------------------------------------------------------
    bert_model, bert_tok, bert_val = train_bert_model(
    CONFIG["BERT_MODEL"], "BERT (bert-base-uncased)", "bert",
    X_train, y_train, X_val, y_val        # WITH expansion
    )
    val_results.append(bert_val)

    bert_test, b_proba, b_labels = evaluate_bert_on_test(
        "BERT", "bert", bert_model, bert_tok, X_test, y_test
    )
    test_results.append(bert_test)
    fpr, tpr, _ = roc_curve(b_labels, b_proba)
    roc_data.append(("BERT", fpr, tpr, roc_auc_score(b_labels, b_proba)))

    # ------------------------------------------------------------------
    # Model 4: ClinicalBERT
    # ------------------------------------------------------------------
    cbert_model, cbert_tok, cbert_val = train_bert_model(
    CONFIG["CLINICAL_MODEL"], "ClinicalBERT (Bio_ClinicalBERT)", "clinicalbert",
    X_train_ne, y_train, X_val_ne, y_val  # WITHOUT expansion
    )
    val_results.append(cbert_val)

    cbert_test, cb_proba, cb_labels = evaluate_bert_on_test(
    "ClinicalBERT", "clinicalbert", cbert_model, cbert_tok, X_test_ne, y_test
    )
    test_results.append(cbert_test)
    fpr, tpr, _ = roc_curve(cb_labels, cb_proba)
    roc_data.append(("ClinicalBERT", fpr, tpr, roc_auc_score(cb_labels, cb_proba)))

    threshold_analysis("ClinicalBERT", cb_labels, cb_proba, "clinicalbert")

    # ------------------------------------------------------------------
    # Model 5: BioMed RoBERTa
    # ------------------------------------------------------------------
    roberta_model, roberta_tok, roberta_val = train_bert_model(
    CONFIG["ROBERTA_MODEL"], "BioMed-RoBERTa", "roberta",
    X_train_ne, y_train, X_val_ne, y_val  # WITHOUT expansion
    )

    val_results.append(roberta_val)

    roberta_test, rb_proba, rb_labels = evaluate_bert_on_test(
    "BioMed-RoBERTa", "roberta", roberta_model, roberta_tok, X_test_ne, y_test
    )
    test_results.append(roberta_test)
    fpr, tpr, _ = roc_curve(rb_labels, rb_proba)
    roc_data.append(("BioMed-RoBERTa", fpr, tpr, roc_auc_score(rb_labels, rb_proba)))

    # ------------------------------------------------------------------
    # Comparison & export
    # ------------------------------------------------------------------
    plot_model_comparison(val_results)
    plot_roc_curves(roc_data)
    save_results(val_results, test_results)

    print("\n" + "#" * 60)
    print("PIPELINE COMPLETE- all outputs saved.")
    print("#" * 60)


if __name__ == "__main__":
    main()


############################################################
GOUT ED CHIEF COMPLAINT- NLP CLASSIFICATION PIPELINE
############################################################
Combined dataset: 8,437 rows
Raw Consensus counts:
Consensus
-    7976
N     350
Y      95
U      16
Name: count, dtype: int64
Raw Predict counts:
Predict
N    8168
U     156
Y     111
-       2
Name: count, dtype: int64

After fallback labeling: 8,437 rows
Label distribution:
label
0    8325
1     112
Name: count, dtype: int64
Imbalance ratio: 74.3:1

EXPLORATORY DATA ANALYSIS

Class distribution:
  Non-Gout (0): 8,325  (98.7%)
  Gout (1): 112  (1.3%)

Text length statistics (characters):
        count   mean   std   min   25%    50%    75%    max
label                                                      
0      8325.0  108.3  52.5   3.0  68.0  100.0  141.0  266.0
1       112.0   94.3  41.0  20.0  63.8   88.5  116.8  235.0

Top 20 words (gout class):
  gout: 139
  pain: 108
  pmh: 58
  htn: 45
  foot: 36
  swel

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/656M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: allenai/biomed_roberta_base
Key                         | Status     | 
----------------------------+------------+-
lm_head.decoder.weight      | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/656M [00:00<?, ?B/s]

Class weights- Non-Gout: 1.0, Gout: 10.0
  Epoch 1/5 | Loss: 0.3944 | Val F1: 0.0000
  Epoch 2/5 | Loss: 0.1890 | Val F1: 0.6500
  Epoch 3/5 | Loss: 0.1423 | Val F1: 0.7586
  Epoch 4/5 | Loss: 0.1038 | Val F1: 0.7333
  Epoch 5/5 | Loss: 0.0958 | Val F1: 0.7333
Best model saved → /content/drive/MyDrive/gout_outputs/roberta_best.pt

RESULTS: BioMed-RoBERTa
              precision    recall  f1-score   support

    Non-Gout       1.00      1.00      1.00      1249
        Gout       0.92      0.65      0.76        17

    accuracy                           0.99      1266
   macro avg       0.96      0.82      0.88      1266
weighted avg       0.99      0.99      0.99      1266

  F1 (Gout):        0.7586
  Precision (Gout): 0.9167
  Recall (Gout):    0.6471
  ROC-AUC:          0.9937

RESULTS: BioMed-RoBERTa [TEST]
              precision    recall  f1-score   support

    Non-Gout       1.00      1.00      1.00      1249
        Gout       0.75      0.71      0.73        17

    accuracy